# Stage 1 · Tissue clustering → coarse anchors

Cluster each tissue independently and identify Seurat anchors across its two donors (`find_anchor`), separately for mCG (5-kb LSI) and 3C (100-kb PCA). Example tissue: **TrCo**.

These are the original pipeline notebooks, concatenated in execution order with paths normalized to `ENTEX_ROOT`. They document the method and run per tissue / major type across the full raw dataset (heavy compute); they are shown for reference and are **not re-executed in the book**. Example group shown where templated.

In [1]:
# === Reproduction setup ===
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT); import repro_guard


## 1a · mCG: 5-kb LSI + cross-donor anchor identification

_Source: `clustering/tissue/L1/TrCo/_01.5kCG_clustering.ipynb`_

In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

import anndata
import scanpy as sc
import scanpy.external as sce
from sklearn.preprocessing import normalize

from ALLCools.clustering import *
from ALLCools.plot import *
from ALLCools.integration.seurat_class import SeuratIntegration

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'


In [3]:
def dump_embedding(adata, name, n_dim=2):
    # put manifold coordinates into adata.obs
    for i in range(n_dim):
        adata.obs[f'{name}_{i}'] = adata.obsm[f'X_{name}'][:, i]
    return adata


In [4]:
group_name = 'B'


In [5]:
# Parameters
cpu = 5
group_name = "TrCo"
mem_gb = 20


In [6]:
t = group_name
indir = ff'{ENTEX_ROOT}/tissue/{t}/'


In [7]:
mcad = anndata.read_h5ad(f'{indir}data/mCG_5kb.h5ad')
mcad


In [8]:
remove_chromosomes(mcad, exclude_chromosomes=['chrX', 'chrY', 'chrM'])
filter_regions(mcad, n_cell=5)
remove_black_list_region(mcad, black_list_path=f'{REF_ROOT}/blacklist/hg38-blacklist.v2.bed.gz')


In [9]:
model = LSI(scale_factor=10000,
            n_components=50,
            algorithm='arpack',
            random_state=0)


In [10]:
model.fit_transform(mcad, obsm_name='lsi_all')


In [11]:
npc = significant_pc_test(mcad, p_cutoff=0.05, obsm='lsi_all', update=False)


In [12]:
meta = pd.read_csv(f'{ENTEX_ROOT}/clustering/merged/cell_86689_16tissue_meta.csv.gz', header=0, index_col=0)
mcad.obs = meta.loc[mcad.obs.index]


In [13]:
mcad.obsm['X_lsi'] = normalize(mcad.obsm['lsi_all'][:, :npc], axis=1)
tsne(mcad, obsm='X_lsi', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
mcad.obsm[f'5kCG_u{npc}_tsne'] = mcad.obsm['X_tsne'].copy()


In [14]:
ds = 1
coord_base = 'tsne'
mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'5kCG_u{npc}_{coord_base}'].copy()
dump_embedding(mcad, coord_base)

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=300, constrained_layout=True)

tmp = mcad.obs.copy()
ax = axes[0, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab10',
                        scatter_kws={'rasterized':True},
                        show_legend=True)

ax = axes[1, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='celltype',
                        text_anno='celltype', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)

ax = axes[0, 1]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue='mCGFrac',
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )

ax = axes[1, 1]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue=np.log10(tmp['FinalmCReads']+1),
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )


In [15]:
sample_list = mcad.obs['Donor'].unique()
adata_list = [mcad[mcad.obs['Donor']==xx] for xx in sample_list]


In [16]:
integrator = SeuratIntegration()
integrator.find_anchor(adata_list,
                        k_local=None,
                        key_local='X_lsi',
                        k_anchor=5,
                        key_anchor='X',
                        dim_red='lsi-cca',
                        max_cc_cells=50000,
                        k_score=30,
                        k_filter=None,
                        scale_list=[False, False],
                        n_components=npc,
                        n_features=200)
corrected = integrator.integrate(key_correct='lsi_all',
                                 row_normalize=True,
                                 n_components=npc,
                                 k_weight=100,
                                 sd=1)


In [17]:
corrected = pd.DataFrame(normalize(np.concatenate(corrected, axis=0), axis=1), 
                         index=np.concatenate([xx.obs.index for xx in adata_list]))

mcad.obsm[f'lsi_seurat_cc{npc}'] = corrected.loc[mcad.obs.index].values
mcad.obsm['X_lsi'] = normalize(mcad.obsm[f'lsi_seurat_cc{npc}'][:, :npc], axis=1)

tsne(mcad, obsm='X_lsi', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
mcad.obsm[f'lsi_u{npc}_seurat_cc{npc}_tsne'] = mcad.obsm['X_tsne'].copy()


In [18]:
ds = 1
coord_base = 'tsne'
mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'lsi_u{npc}_seurat_cc{npc}_tsne'].copy()
dump_embedding(mcad, coord_base)

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=300, constrained_layout=True)

tmp = mcad.obs.copy()
ax = axes[0, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab10',
                        scatter_kws={'rasterized':True},
                        show_legend=True)

ax = axes[1, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='celltype',
                        text_anno='celltype', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)

ax = axes[0, 1]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue='mCGFrac',
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )

ax = axes[1, 1]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue=np.log10(tmp['FinalmCReads']+1),
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )


In [19]:
mcad.write_h5ad('mCG_5kb_lsi.h5ad')


In [20]:
integrator.anchor[(0,1)].to_hdf(f'mCG_5kb_seurat_anchor_{"_".join(sample_list)}.hdf', key='data')


In [21]:
fig, ax = plt.subplots(figsize=(3,2), dpi=300)
sns.histplot(integrator.anchor[(0,1)]['score'], bins=50, ax=ax)


In [22]:
fig, axes = plt.subplots(2, 2, figsize=(10,8), dpi=300)

mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'lsi_u{npc}_seurat_cc{npc}_tsne'].copy()
dump_embedding(mcad, coord_base)
tmp = mcad.obs.copy()
anchor1 = mcad.obs.loc[adata_list[0].obs.index, ['tsne_0', 'tsne_1']].iloc[integrator.anchor[(0,1)]['x1']].values
anchor2 = mcad.obs.loc[adata_list[1].obs.index, ['tsne_0', 'tsne_1']].iloc[integrator.anchor[(0,1)]['x2']].values

for k,ax in zip([0, 0.5, 0.8, 1.0], axes.flatten()):
    
    _ = categorical_scatter(data=tmp,
                            ax=ax,
                            coord_base=coord_base,
                            hue='celltype',
                            # text_anno='celltype', 
                            s=ds,
                            labelsize=8,
                            max_points=None,
                            palette='tab20',
                            scatter_kws={'rasterized':True},
                            # legend_kws={'ncol':2}, 
                            show_legend=False)

    sela = (integrator.anchor[(0,1)]['score']>=k)
    print(sela.sum())
    anchor1_tmp = anchor1[sela]
    anchor2_tmp = anchor2[sela]
    for xx,yy in zip(anchor1_tmp, anchor2_tmp):
        ax.plot([xx[0], yy[0]], [xx[1], yy[1]], color='k', alpha=1.0, linewidth=0.1)


In [23]:
fig, axes = plt.subplots(2, 2, figsize=(10,8), dpi=300)

mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'5kCG_u{npc}_{coord_base}'].copy()
dump_embedding(mcad, coord_base)
tmp = mcad.obs.copy()
anchor1 = mcad.obs.loc[adata_list[0].obs.index, ['tsne_0', 'tsne_1']].iloc[integrator.anchor[(0,1)]['x1']].values
anchor2 = mcad.obs.loc[adata_list[1].obs.index, ['tsne_0', 'tsne_1']].iloc[integrator.anchor[(0,1)]['x2']].values

for k,ax in zip([0, 0.5, 0.8, 1.0], axes.flatten()):
    
    _ = categorical_scatter(data=tmp,
                            ax=ax,
                            coord_base=coord_base,
                            hue='celltype',
                            # text_anno='celltype', 
                            s=ds,
                            labelsize=8,
                            max_points=None,
                            palette='tab20',
                            scatter_kws={'rasterized':True},
                            # legend_kws={'ncol':2}, 
                            show_legend=False)

    sela = (integrator.anchor[(0,1)]['score']>=k)
    print(sela.sum())
    anchor1_tmp = anchor1[sela]
    anchor2_tmp = anchor2[sela]
    for xx,yy in zip(anchor1_tmp, anchor2_tmp):
        ax.plot([xx[0], yy[0]], [xx[1], yy[1]], color='k', alpha=1.0, linewidth=0.1)


## Geosketch

In [24]:
tmp = anndata.read_h5ad('mCG_5kb_lsi.h5ad')
tmp

In [25]:
mcad.obs = tmp.obs.copy()
mcad.obsm = tmp.obsm.copy()

In [26]:
npc = significant_pc_test(mcad, p_cutoff=0.05, obsm='lsi_all', update=False)
mcad.obsm['X_lsi'] = normalize(mcad.obsm['lsi_all'][:, :npc], axis=1)


In [27]:
from geosketch import gs

sketch_index = gs(mcad.obsm['X_lsi'], 2000, replace=False)


In [28]:
ds = 1
coord_base = 'tsne'
mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'5kCG_u{npc}_tsne'].copy()
dump_embedding(mcad, coord_base)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=300, constrained_layout=True)

tmp = mcad.obs.iloc[sketch_index].copy()
ax = axes[0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='celltype',
                        text_anno='celltype', 
                        s=ds*4,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)

tmp = mcad.obs.copy()
ax = axes[1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='celltype',
                        text_anno='celltype', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)



In [29]:
model = LSI(scale_factor=10000,
            n_components=50,
            algorithm='arpack',
            random_state=0)


In [30]:
model.fit(mcad[sketch_index])


In [31]:
model.transform(mcad, obsm_name='lsi_geosk_all')


In [32]:
npc = significant_pc_test(mcad, p_cutoff=0.05, obsm='lsi_geosk_all', update=False)


In [33]:
mcad.obsm['X_geosk_lsi'] = normalize(mcad.obsm['lsi_geosk_all'][:, :npc], axis=1)
tsne(mcad, obsm='X_geosk_lsi', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
mcad.obsm[f'5kCG_geosk_u{npc}_tsne'] = mcad.obsm['X_tsne'].copy()


In [34]:
ds = 1
coord_base = 'tsne'
mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'5kCG_geosk_u{npc}_{coord_base}'].copy()
dump_embedding(mcad, coord_base)

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=300, constrained_layout=True)

tmp = mcad.obs.copy()
ax = axes[0, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab10',
                        scatter_kws={'rasterized':True},
                        show_legend=True)

ax = axes[1, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='celltype',
                        text_anno='celltype', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)

ax = axes[0, 1]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue='mCGFrac',
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )

ax = axes[1, 1]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue=np.log10(tmp['FinalmCReads']+1),
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )


In [35]:
npc = significant_pc_test(mcad, p_cutoff=0.05, obsm='lsi_all', update=False)
mcad.obsm['X_lsi'] = normalize(mcad.obsm['lsi_all'][:, :npc], axis=1)


In [36]:
sample_list = mcad.obs['Donor'].unique()
adata_list = [mcad[mcad.obs['Donor']==xx] for xx in sample_list]


In [37]:
anchor = pd.read_hdf(f'mCG_5kb_seurat_anchor_{"_".join(sample_list)}.hdf', key='data')
coord1 = mcad.obsm['X_lsi'][mcad.obs['Donor']==sample_list[0]][anchor['x1']]
coord2 = mcad.obsm['X_lsi'][mcad.obs['Donor']==sample_list[1]][anchor['x2']]
anchor['dist_pc'] = np.linalg.norm((coord1 - coord2), ord=2, axis=1)

coord1 = mcad.obsm['X_geosk_lsi'][mcad.obs['Donor']==sample_list[0]][anchor['x1']]
coord2 = mcad.obsm['X_geosk_lsi'][mcad.obs['Donor']==sample_list[1]][anchor['x2']]
anchor['dist_geoskpc'] = np.linalg.norm((coord1 - coord2), ord=2, axis=1)

anchor['ct1'] = adata_list[0].obs['celltype'].values[anchor['x1']].astype(str)
anchor['ct2'] = adata_list[1].obs['celltype'].values[anchor['x2']].astype(str)


In [38]:
fig, axes = plt.subplots(3, 1, figsize=(3,4), dpi=300, constrained_layout=True, sharey='all')
for xx,ax in zip(['dist_pc', 'dist_geoskpc', 'score'], axes):
    sns.histplot(anchor.loc[anchor['ct1']==anchor['ct2'], xx], bins=100, ax=ax)
    sns.histplot(anchor.loc[anchor['ct1']!=anchor['ct2'], xx], bins=100, ax=ax)
    for t in [0, 1, 5, 10, 20, 40, 60, 80, 90, 95, 99, 100]:
        pos = np.percentile(anchor[xx], t)
        ax.plot([pos, pos], [0, anchor.shape[0]/100], 'r')


## 1b · 3C: 100-kb PCA + cross-donor anchor identification

_Source: `clustering/tissue/L1/template/_01b.100k3C_clustering.ipynb`_

In [39]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

import anndata
import scanpy as sc
import scanpy.external as sce
from sklearn.preprocessing import normalize
from sklearn.decomposition import TruncatedSVD

from ALLCools.clustering import *
from ALLCools.plot import *
from ALLCools.integration.seurat_class import SeuratIntegration

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'


In [40]:
def dump_embedding(adata, name, n_dim=2):
    # put manifold coordinates into adata.obs
    for i in range(n_dim):
        adata.obs[f'{name}_{i}'] = adata.obsm[f'X_{name}'][:, i]
    return adata


In [41]:
group_name = 'EG'


In [42]:
# Parameters
cpu = 5
group_name = "TrCo"
mem_gb = 20


In [43]:
t = group_name
indir = ff'{ENTEX_ROOT}/tissue/{t}/data/'


## HiC embedding

In [44]:
chromsize = pd.read_csv(f'{REF_ROOT}/hg38/fasta/hg38.main.chrom.sizes', sep='\t', index_col=0, header=None)
chromsize = chromsize.iloc[:22]
chrom = chromsize.index


In [45]:
meta = pd.read_csv(f'{ENTEX_ROOT}/clustering/merged/cell_86689_16tissue_meta.csv.gz', header=0, index_col=0)
cell_list = pd.read_csv(f'{indir}cell_table.tsv', sep='\t', index_col=0, header=None)
meta = meta.loc[cell_list.index]
sample_list = np.sort(meta['Donor'].unique())
selc = (meta['Donor']==sample_list[0])


In [46]:
matrix = []
C = np.zeros((selc.sum(), meta.shape[0]-selc.sum()))
dim = 50

for c in chrom:
    tmp = np.load(f'{indir}raw/{c}.npz')['arr_0']
    
    ## chromosome pca
    dim = min(dim, tmp.shape[0] - 1, tmp.shape[1] - 1)
    model = TruncatedSVD(n_components=dim, algorithm='arpack')
    tmp_reduce = model.fit_transform(tmp)
    matrix.append(tmp_reduce / model.singular_values_)
    
    ## raw feature cca
    C += tmp[selc].dot(tmp[~selc].T)
    print(c)
    

In [47]:
matrix = np.concatenate(matrix, axis=1)
mcad = anndata.AnnData(X=matrix, obs=meta)


In [48]:
# mcad = anndata.read_h5ad('HiC_100kb_pca.h5ad')


In [49]:
tmp = anndata.read_h5ad(f'{ENTEX_ROOT}/clustering/merged/cell_86689_16tissue_100k3C_autosomal.h5ad')
mcad.obs['ClusterTissue'] = tmp.obs.loc[mcad.obs.index, 'ClusterTissue'].copy()


In [50]:
dim = 50
model = TruncatedSVD(n_components=dim, algorithm='arpack')
mcad.obsm['100k3C_pca'] = model.fit_transform(mcad.X)
mcad.uns['100k3C_eigen'] = model.singular_values_.copy()
# mcad.obsm['100k3C_u'] = mcad.obsm['100k3C_pca'] / model.singular_values_


In [51]:
npc = significant_pc_test(mcad, p_cutoff=0.05, update=False, obsm='100k3C_pca')


In [52]:
mcad.obsm['X_pca'] = normalize(mcad.obsm['100k3C_pca'][:, :npc], axis=1)
tsne(mcad, obsm='X_pca', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
mcad.obsm[f'100k3C_pc{npc}_tsne'] = mcad.obsm['X_tsne'].copy()


In [53]:
ds = 1
coord_base = 'tsne'
mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'100k3C_pc{npc}_{coord_base}'].copy()
dump_embedding(mcad, coord_base)

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=300, constrained_layout=True)

tmp = mcad.obs.copy()
ax = axes[0, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab10',
                        scatter_kws={'rasterized':True},
                        show_legend=True)

ax = axes[1, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='celltype',
                        text_anno='celltype', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)

ax = axes[0, 1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='ClusterTissue',
                        text_anno='ClusterTissue', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)

ax = axes[1, 1]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue=np.log10(tmp['CisLongContact']+1),
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )


## Find anchor with raw feature CCA

In [54]:
model = TruncatedSVD(n_components=50, algorithm='arpack', random_state=0)
U = model.fit_transform(C)
sel_dim = model.singular_values_ != 0
V = model.components_[sel_dim].T
U = U[:, sel_dim] / model.singular_values_[sel_dim]


In [55]:
corrected = pd.DataFrame(normalize(np.concatenate([U, V], axis=0), axis=1), 
                         index=np.concatenate([mcad.obs.index[selc], mcad.obs.index[~selc]]))
mcad.obsm['X_cca'] = corrected.loc[mcad.obs.index].values[:, :npc]


In [56]:
adata_list = [mcad[mcad.obs['Donor']==xx] for xx in sample_list]


In [57]:
integrator = SeuratIntegration()
integrator.find_anchor(adata_list,
                        k_local=None,
                        key_local='X_pca',
                        k_anchor=5,
                        key_anchor='X_cca',
                        dim_red='pca',
                        max_cc_cells=50000,
                        k_score=30,
                        k_filter=None,
                        scale_list=[False, False],
                        n_components=npc,
                        n_features=200)
corrected = integrator.integrate(key_correct='100k3C_pca',
                                 row_normalize=True,
                                 n_components=npc,
                                 k_weight=100,
                                 sd=1)


In [58]:
corrected = pd.DataFrame(normalize(np.concatenate(corrected, axis=0), axis=1), 
                         index=np.concatenate([xx.obs.index for xx in adata_list]))

mcad.obsm[f'100k3C_pc{npc}_seuratrawcc{npc}'] = corrected.loc[mcad.obs.index].values
mcad.obsm[f'100k3C_pc{npc}_seuratrawcc{npc}'] = normalize(mcad.obsm[f'100k3C_pc{npc}_seuratrawcc{npc}'][:, :npc], axis=1)

tsne(mcad, obsm=f'100k3C_pc{npc}_seuratrawcc{npc}', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
mcad.obsm[f'100k3C_pc{npc}_seuratrawcc{npc}_tsne'] = mcad.obsm['X_tsne'].copy()


In [59]:
ds = 1
coord_base = 'tsne'
mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'100k3C_pc{npc}_seuratrawcc{npc}_tsne'].copy()
dump_embedding(mcad, coord_base)

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=300, constrained_layout=True)

tmp = mcad.obs.copy()
ax = axes[0, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab10',
                        scatter_kws={'rasterized':True},
                        show_legend=True)

ax = axes[1, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='celltype',
                        text_anno='celltype', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)

ax = axes[0, 1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='ClusterTissue',
                        text_anno='ClusterTissue', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)

ax = axes[1, 1]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue=np.log10(tmp['CisLongContact']+1),
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )


## Find anchor with chromosome U

In [60]:
integrator = SeuratIntegration()
integrator.find_anchor(adata_list,
                        k_local=None,
                        key_local='X_pca',
                        k_anchor=5,
                        key_anchor='X',
                        dim_red='pca-cca',
                        max_cc_cells=50000,
                        k_score=30,
                        k_filter=None,
                        scale_list=[False, False],
                        n_components=npc,
                        n_features=200)


In [61]:
corrected = integrator.integrate(key_correct='100k3C_pca',
                                 row_normalize=True,
                                 n_components=npc,
                                 k_weight=100,
                                 sd=1)


In [62]:
anchor = integrator.anchor[(0,1)].copy()
anchor_file = f'HiC_100kb_seurat_anchor_{"_".join(sample_list)}.hdf'
anchor.to_hdf(anchor_file, key='data')


In [63]:
mcad.obsm['X_pca'] = normalize(mcad.obsm['100k3C_pca'][:, :npc], axis=1)
coord1 = mcad.obsm['X_pca'][mcad.obs['Donor']==sample_list[0]][anchor['x1']]
coord2 = mcad.obsm['X_pca'][mcad.obs['Donor']==sample_list[1]][anchor['x2']]
anchor['dist_pc'] = np.linalg.norm((coord1 - coord2), ord=2, axis=1)


In [64]:
corrected = pd.DataFrame(normalize(np.concatenate(corrected, axis=0), axis=1), 
                         index=np.concatenate([xx.obs.index for xx in adata_list]))

mcad.obsm[f'100k3C_pc{npc}_seuratcc{npc}'] = corrected.loc[mcad.obs.index].values
mcad.obsm[f'100k3C_pc{npc}_seuratcc{npc}'] = normalize(mcad.obsm[f'100k3C_pc{npc}_seuratcc{npc}'][:, :npc], axis=1)

tsne(mcad, obsm=f'100k3C_pc{npc}_seuratcc{npc}', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
mcad.obsm[f'100k3C_pc{npc}_seuratcc{npc}_tsne'] = mcad.obsm['X_tsne'].copy()


In [65]:
ds = 1
coord_base = 'tsne'
mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'100k3C_pc{npc}_seuratcc{npc}_tsne'].copy()
dump_embedding(mcad, coord_base)

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=300, constrained_layout=True)

tmp = mcad.obs.copy()
ax = axes[0, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab10',
                        scatter_kws={'rasterized':True},
                        show_legend=True)

ax = axes[1, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='celltype',
                        text_anno='celltype', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)

ax = axes[0, 1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='ClusterTissue',
                        text_anno='ClusterTissue', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)


ax = axes[1, 1]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue=np.log10(tmp['CisLongContact']+1),
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )


## Harmony

In [66]:
sce.pp.harmony_integrate(mcad, key='Donor', basis='X_pca', adjusted_basis=f'100k3C_pc{npc}_hm', max_iter_harmony=30, random_state=0)
mcad.obsm[f'100k3C_pc{npc}_hm'] = normalize(mcad.obsm[f'100k3C_pc{npc}_hm'], axis=1)
tsne(mcad, obsm=f'100k3C_pc{npc}_hm', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
mcad.obsm[f'100k3C_pc{npc}_hm_tsne'] = mcad.obsm['X_tsne'].copy()


In [67]:
ds = 1
coord_base = 'tsne'
mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'100k3C_pc{npc}_hm_tsne'].copy()
dump_embedding(mcad, coord_base)

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=300, constrained_layout=True)

tmp = mcad.obs.copy()
ax = axes[0, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab10',
                        scatter_kws={'rasterized':True},
                        show_legend=True)

ax = axes[1, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='celltype',
                        text_anno='celltype', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)

ax = axes[0, 1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='ClusterTissue',
                        text_anno='ClusterTissue', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)

ax = axes[1, 1]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue=np.log10(tmp['FinalmCReads']+1),
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )


In [68]:
import snapatac2 as snap

snap.pp.mnc_correct(mcad, batch='Donor', use_rep='X_pca', key_added=f'100k3C_pc{npc}_mnn')
mcad.obsm[f'100k3C_pc{npc}_mnn'] = normalize(mcad.obsm[f'100k3C_pc{npc}_mnn'], axis=1)
tsne(mcad, obsm=f'100k3C_pc{npc}_mnn', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
mcad.obsm[f'100k3C_pc{npc}_mnn_tsne'] = mcad.obsm['X_tsne'].copy()


In [69]:
ds = 1
coord_base = 'tsne'
mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'100k3C_pc{npc}_mnn_tsne'].copy()
dump_embedding(mcad, coord_base)

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=300, constrained_layout=True)

tmp = mcad.obs.copy()
ax = axes[0, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab10',
                        scatter_kws={'rasterized':True},
                        show_legend=True)

ax = axes[1, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='celltype',
                        text_anno='celltype', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)

ax = axes[0, 1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='ClusterTissue',
                        text_anno='ClusterTissue', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)

ax = axes[1, 1]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue=np.log10(tmp['FinalmCReads']+1),
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )


## Find anchor with chromosome U

In [70]:
# integrator.anchor[(0,1)].to_hdf(f'HiC_100kb_seurat_anchor_{"_".join(sample_list)}.hdf', key='data')


In [71]:
mcad.obsm['X_pca'] = normalize(mcad.obsm['100k3C_pca'][:, :npc] / mcad.uns['100k3C_eigen'][:npc], axis=1)
corrected = integrator.integrate(key_correct='X_pca',
                                 row_normalize=True,
                                 n_components=npc,
                                 k_weight=100,
                                 sd=1)


In [72]:
coord1 = mcad.obsm['X_pca'][mcad.obs['Donor']==sample_list[0]][anchor['x1']]
coord2 = mcad.obsm['X_pca'][mcad.obs['Donor']==sample_list[1]][anchor['x2']]
anchor['dist_u'] = np.linalg.norm((coord1 - coord2), ord=2, axis=1)


In [73]:
corrected = pd.DataFrame(normalize(np.concatenate(corrected, axis=0), axis=1), 
                         index=np.concatenate([xx.obs.index for xx in adata_list]))

mcad.obsm[f'100k3C_u{npc}_seuratcc{npc}'] = corrected.loc[mcad.obs.index].values
mcad.obsm[f'100k3C_u{npc}_seuratcc{npc}'] = normalize(mcad.obsm[f'100k3C_u{npc}_seuratcc{npc}'][:, :npc], axis=1)

tsne(mcad, obsm=f'100k3C_u{npc}_seuratcc{npc}', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
mcad.obsm[f'100k3C_u{npc}_seuratcc{npc}_tsne'] = mcad.obsm['X_tsne'].copy()


In [74]:
ds = 1
coord_base = 'tsne'
mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'100k3C_u{npc}_seuratcc{npc}_tsne'].copy()
dump_embedding(mcad, coord_base)

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=300, constrained_layout=True)

tmp = mcad.obs.copy()
ax = axes[0, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab10',
                        scatter_kws={'rasterized':True},
                        show_legend=True)

ax = axes[1, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='celltype',
                        text_anno='celltype', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)

ax = axes[0, 1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='ClusterTissue',
                        text_anno='ClusterTissue', 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        legend_kws={'ncol':1}, 
                        show_legend=True)


ax = axes[1, 1]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue=np.log10(tmp['CisLongContact']+1),
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )


In [75]:
anchor.to_hdf(anchor_file, key='data')


In [76]:
fig, ax = plt.subplots(figsize=(3,2), dpi=300)
xx, yy = anchor['dist_pc'], anchor['dist_u']
ax.scatter(xx, yy, c='k', edgecolor='none', s=0.5, alpha=1, rasterized=True)
ax.plot([np.min(xx), np.max(xx)], [np.min(xx), np.max(xx)])
ax.set_xlabel('Dist PC')
ax.set_ylabel('Dist U')


In [77]:
mcad.obs.index.name = 'cell'
# mcad = anndata.AnnData(obs=mcad.obs, obsm=mcad.obsm)
mcad.write_h5ad('HiC_100kb_pca.h5ad')


In [78]:
mcad